# Lesson 16: On-Device Model Context Protocol (MCP) — Connecting Local SLMs to STDIO & HTTP Tool Servers

In this companion notebook, you will:
1. Install the open-source Python `mcp` SDK (`FastMCP`) on Google Colab.
2. Build a lightweight local MCP server exposing edge device diagnostics (`get_system_health`, `query_edge_logs`).
3. Connect an in-process MCP client session to discover tools (`tools/list`) and invoke them (`tools/call`).

In [ ]:
!pip install -q -U mcp

## 1. Define a FastMCP Edge Diagnostics Server
Save a clean `stdio` MCP server script (`edge_mcp_server.py`) that any local SLM host (Ollama, llama.cpp, or VS Code) can launch as an isolated subprocess.

In [ ]:
server_code = '''from mcp.server.fastmcp import FastMCP

mcp = FastMCP("EdgeDiagnosticsServer")

@mcp.tool()
def get_device_telemetry(node_id: str) -> dict:
    """Return CPU temperature and memory headroom for an edge node."""
    return {"node_id": node_id, "cpu_temp_c": 52.3, "free_ram_mb": 2480}

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''
with open("edge_mcp_server.py", "w") as f:
    f.write(server_code)
print("Wrote edge_mcp_server.py")

## 2. Connect via Python MCP Client Over `stdio`
Launch the server subprocess, negotiate capabilities, list tools, and execute `get_device_telemetry`.

In [ ]:
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def run_mcp_demo():
    params = StdioServerParameters(command="python", args=["edge_mcp_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print("Discovered MCP Tools:", [t.name for t in tools.tools])
            res = await session.call_tool("get_device_telemetry", {"node_id": "jetson-04"})
            print("Tool Result:", res.content[0].text)

await run_mcp_demo()